# Unconstrained Optimization

## Data

All the analysis below applies to the data set,
* `data/spx_returns_weekly.xlsx`
* The file has **weekly** returns.
* For annualization, use 52 periods per year.

Consider only the following 10 stocks...

In [1]:
TICKS =  ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','BRK/B','LLY']

As well as the ETF,

In [2]:
TICK_ETF = 'SPY'

### Data Processing

In [3]:
import numpy as np
import pandas as pd

In [4]:
INFILE = 'FINM Portfolio/spx_returns_weekly.xlsx'
SHEET_INFO = 'spx names'
SHEET_RETURNS = 'spx returns'
SHEET_BENCH = 'additional returns'

FREQ = 52

In [5]:
info = pd.read_excel(INFILE,sheet_name=SHEET_INFO)
info.set_index('ticker',inplace=True)
info.loc[TICKS].rename(columns={
    'name': 'Name',
    'gics_sector_name': 'Sector',
    'mkt cap': 'Market capitalization',
})

,Name,Sector,Market capitalization
ticker,,,
AAPL,Apple Inc,Information Technology,4.631217e+12
NVDA,NVIDIA Corp,Information Technology,5.105232e+12
MSFT,Microsoft Corp,Information Technology,2.860690e+12
GOOGL,Alphabet Inc,Communication Services,4.335816e+12
AMZN,Amazon.com Inc,Consumer Discretionary,2.639149e+12
META,Meta Platforms Inc,Communication Services,1.698738e+12
TSLA,Tesla Inc,Consumer Discretionary,1.531434e+12
AVGO,Broadcom Inc,Information Technology,1.902889e+12
BRK/B,Berkshire Hathaway Inc,Financials,1.064448e+12


In [6]:
rets = pd.read_excel(INFILE,sheet_name=SHEET_RETURNS)
rets.set_index('date',inplace=True)
rets = rets[TICKS]

In [7]:
bench = pd.read_excel(INFILE,sheet_name=SHEET_BENCH)
bench.set_index('date',inplace=True)
rets[TICK_ETF] = bench[TICK_ETF]

***

# 1. Risk Statistics

### 1.1.

Display a table with the following metrics for each of the return series.

* mean (annualized)
* volatility (annualized)
* Sharpe ratio (annualized)
* skewness
* kurtosis
* maximum drawdown

#### Note
We  have total returns, and Sharpe ratio is technically defined for excess returns. Don't worry about the difference. (Or subtract `SHV` if you prefer.)

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
from IPython.display import display
import statsmodels.api as sm

### Used ChatGPT to help summarize the performance and risk metrics.
### Used helper functions from an earlier class taken in the summer below.

INFILE = 'FINM Portfolio/spx_returns_weekly.xlsx'


rets = pd.read_excel(
    INFILE,
    sheet_name=SHEET_RETURNS,
    index_col="date",
    parse_dates=True
)[TICKS]

bench = pd.read_excel(
    INFILE,
    sheet_name=SHEET_BENCH,
    index_col="date",
    parse_dates=True
)

rets[TICK_ETF] = bench[TICK_ETF]
rets = rets[TICKS + [TICK_ETF]].sort_index().dropna()

def calc_return_metrics(data, as_df=False, adj=12):
    """
    Calculate return metrics for a given dataset. Specifically:
    - Annualized Return
    - Annualized Volatility
    - Annualized Sharpe Ratio
    - Annualized Sortino Ratio (not part of the course, but useful to know)

    Args:
        data : Returns time series.
        as_df (bool, optional): Return a df or dict. Defaults to False.
        adj (int, optional): Annualization. Defaults to 12.

    Returns:
        DataFrame or dict: Summary of return metrics.
    """
    summary = dict()
    summary["Annualized Return"] = data.mean() * adj
    summary["Annualized Volatility"] = data.std() * np.sqrt(adj)
    summary["Annualized Sharpe Ratio"] = (
        summary["Annualized Return"] / summary["Annualized Volatility"]
    )
    summary["Annualized Sortino Ratio"] = (
        summary["Annualized Return"] * np.sqrt(adj) / (data[data < 0].std())
    )

    return pd.DataFrame(summary, index=data.columns) if as_df else summary

def calc_risk_metrics(data, as_df=False, var=0.05):
    """
    Calculate risk metrics for a DataFrame of assets.

    Args:
        data (pd.DataFrame): DataFrame of asset returns.
        as_df (bool, optional): Return a DF or a dict. Defaults to False.
        adj (int, optional): Annualizatin. Defaults to 12.
        var (float, optional): VaR level. Defaults to 0.05.

    Returns:
        Union[dict, DataFrame]: Dict or DataFrame of risk metrics.
    """
    summary = dict()
    summary["Skewness"] = data.skew()
    summary["Excess Kurtosis"] = data.kurtosis()
    summary[f"VaR ({var})"] = data.quantile(var, axis=0)
    summary[f"CVaR ({var})"] = data[data <= data.quantile(var, axis=0)].mean()
    summary["Min"] = data.min()
    summary["Max"] = data.max()

    wealth_index = 1000 * (1 + data).cumprod()
    previous_peaks = wealth_index.cummax()
    drawdowns = (wealth_index - previous_peaks) / previous_peaks

    summary["Max Drawdown"] = drawdowns.min()

    summary["Bottom"] = drawdowns.idxmin()
    summary["Peak"] = previous_peaks.idxmax()

    recovery_date = []
    for col in wealth_index.columns:
        prev_max = previous_peaks[col][: drawdowns[col].idxmin()].max()
        recovery_wealth = pd.DataFrame([wealth_index[col][drawdowns[col].idxmin() :]]).T
        recovery_date.append(
            recovery_wealth[recovery_wealth[col] >= prev_max].index.min()
        )
    summary["Recovery"] = ["-" if pd.isnull(i) else i for i in recovery_date]

    summary["Duration (days)"] = [
        (i - j).days if i != "-" else "-"
        for i, j in zip(summary["Recovery"], summary["Bottom"])
    ]

    return pd.DataFrame(summary, index=data.columns) if as_df else summary

def calc_perf_metrics(data, adj=12):
    """
    Calculate performance metrics for a given dataset.

    Args:
        data : Returns time series.
        adj (int, optional): Annualization. Defaults to 12.

    Returns:
        DataFrame: Summary of performance metrics.
    """
    summary = {**calc_return_metrics(data, adj), **calc_risk_metrics(data, adj)}
    summary["Calmar Ratio"] = summary["Annualized Return"] / abs(
        summary["Max Drawdown"]
    )
    return pd.DataFrame(summary, index=data.columns)

return_metrics = calc_return_metrics(rets, as_df=True, adj=52)
risk_metrics = calc_risk_metrics(rets, as_df=True)

summary = pd.concat([return_metrics, risk_metrics], axis=1)

summary = summary[
    [
        "Annualized Return",
        "Annualized Volatility",
        "Annualized Sharpe Ratio",
        "Skewness",
        "Excess Kurtosis",
        "Max Drawdown"
    ]
]

display(summary)

,Annualized Return,Annualized Volatility,Annualized Sharpe Ratio,Skewness,Excess Kurtosis,Max Drawdown
AAPL,0.248517,0.277602,0.895227,-0.182144,1.804677,-0.346407
NVDA,0.645347,0.454393,1.420241,0.350905,1.491708,-0.659360
MSFT,0.235299,0.237816,0.989416,0.132668,1.684965,-0.350540
GOOGL,0.269840,0.287311,0.939192,0.532608,3.103957,-0.418641
AMZN,0.261284,0.304504,0.858063,-0.087187,1.341941,-0.548307
META,0.232050,0.357490,0.649108,0.036991,3.490072,-0.760252
TSLA,0.437496,0.579650,0.754758,0.559274,1.633548,-0.722459
AVGO,0.393854,0.382501,1.029681,0.642908,3.272212,-0.400358
BRK/B,0.134847,0.188241,0.716352,-0.186487,2.533935,-0.264770
LLY,0.301432,0.298242,1.010699,0.138271,1.976150,-0.344750


### 1.2.

As a standalone investment, which is most attractive? And least? Justify your answer.

In [9]:
max_sharpe = summary["Annualized Sharpe Ratio"].idxmax()
min_sharpe = summary["Annualized Sharpe Ratio"].idxmin()
print(f"The most attractive standalone investment is {max_sharpe}")
print(f"The least attractive standalone investment is {min_sharpe}")
print(f"SNDK is the most attractive because it has the highest Sharpe Ratio")
print("FDXF is the least attractive because it has the lowest Sharpe Ratio")


The most attractive standalone investment is NVDA
The least attractive standalone investment is META
SNDK is the most attractive because it has the highest Sharpe Ratio
FDXF is the least attractive because it has the lowest Sharpe Ratio


### 1.3.

For each investment, estimate a regression against `SPY`. Report the 
* alpha (annualized as a mean)
* beta
* info ratio
* r-squared

Based on this table, which investment seems most attractive relative to holding `SPY`?

In [10]:
regression_metrics = dict()

# A column of ones allows the regression to estimate an intercept.
X = np.column_stack([np.ones(len(rets)), rets[TICK_ETF]])

for tick in TICKS:
    y = rets[tick]

    # Ordinary least squares: estimate the intercept and slope.
    alpha, beta = np.linalg.lstsq(X, y, rcond=None)[0]

    residuals = y - (alpha + beta * rets[TICK_ETF])

    regression_metrics[tick] = {
        "Annualized Alpha": alpha * FREQ,
        "Beta": beta,
        "Annualized Information Ratio": (
            alpha / residuals.std() * np.sqrt(FREQ)
        ),
        "R-squared": 1 - (
            residuals.pow(2).sum()
            / (y - y.mean()).pow(2).sum()
        )
    }

regression_summary = pd.DataFrame(regression_metrics).T

# SPY against itself: zero alpha and residual risk.
# Its information ratio is undefined because the denominator is zero.
regression_summary.loc[TICK_ETF] = [0.0, 1.0, np.nan, 1.0]

display(regression_summary.round(4))

best_relative = regression_summary.loc[
    TICKS, "Annualized Information Ratio"
].idxmax()

print(f"Most attractive relative to SPY: {best_relative}")

,Annualized Alpha,Beta,Annualized Information Ratio,R-squared
AAPL,0.0865,1.1161,0.4247,0.4618
NVDA,0.3960,1.7179,1.1329,0.4083
MSFT,0.0878,1.0162,0.5337,0.5216
GOOGL,0.1096,1.1039,0.5016,0.4217
AMZN,0.1006,1.1066,0.4189,0.3773
META,0.0604,1.1824,0.2038,0.3125
TSLA,0.1836,1.7490,0.3683,0.2601
AVGO,0.1873,1.4231,0.6297,0.3955
BRK/B,0.0214,0.7818,0.1593,0.4927
LLY,0.2135,0.6058,0.7622,0.1179


Most attractive relative to SPY: NVDA


NVDA seems the most attractive relative to SPY here. It has the highest annualized alpha and information ratio. 

***

# 2. Portfolio Allocation

### 2.1.

Display the correlation matrix of the returns.

* Based on this information, which investment do you anticipate will get extra weight in the portfolio, beyond what it would merit for its mean return?

* Report the maximally correlated assets and the minimally correlated assets.

In [11]:
corr_matrix = rets.corr()
display(corr_matrix.round(3))

# Exclude the diagonal and retain each distinct pair only once.
upper_triangle = np.triu(
    np.ones(corr_matrix.shape, dtype=bool),
    k=1
)

pair_correlations = corr_matrix.where(upper_triangle).stack()

print(
    "Most correlated:",
    pair_correlations.idxmax(),
    round(pair_correlations.max(), 4)
)

print(
    "Least correlated:",
    pair_correlations.idxmin(),
    round(pair_correlations.min(), 4)
)

# Average correlation with the other investments.
average_correlation = corr_matrix.where(
    ~np.eye(len(corr_matrix), dtype=bool)
).mean()

display(
    average_correlation.sort_values().to_frame("Average Correlation")
)

,AAPL,NVDA,MSFT,GOOGL,AMZN,META,TSLA,AVGO,BRK/B,LLY,SPY
AAPL,1.000,0.482,0.554,0.538,0.459,0.406,0.437,0.507,0.405,0.190,0.680
NVDA,0.482,1.000,0.592,0.451,0.546,0.428,0.415,0.573,0.300,0.162,0.639
MSFT,0.554,0.592,1.000,0.607,0.629,0.568,0.408,0.533,0.359,0.255,0.722
GOOGL,0.538,0.451,0.607,1.000,0.591,0.510,0.366,0.477,0.338,0.170,0.649
AMZN,0.459,0.546,0.629,0.591,1.000,0.527,0.395,0.444,0.270,0.144,0.614
META,0.406,0.428,0.568,0.510,0.527,1.000,0.265,0.403,0.275,0.116,0.559
TSLA,0.437,0.415,0.408,0.366,0.395,0.265,1.000,0.373,0.198,0.126,0.510
AVGO,0.507,0.573,0.533,0.477,0.444,0.403,0.373,1.000,0.303,0.104,0.629
BRK/B,0.405,0.300,0.359,0.338,0.270,0.275,0.198,0.303,1.000,0.297,0.702
LLY,0.190,0.162,0.255,0.170,0.144,0.116,0.126,0.104,0.297,1.000,0.343


Most correlated: ('MSFT', 'SPY') 0.7222
Least correlated: ('AVGO', 'LLY') 0.1045


,Average Correlation
LLY,0.190770
BRK/B,0.344661
TSLA,0.349377
META,0.405604
AVGO,0.434587
NVDA,0.458789
AMZN,0.462149
AAPL,0.465800
GOOGL,0.469846
MSFT,0.522662


Here LLY is the clearest candidate for extra weight because of diversification. It's average correlation with the other investments is the lowest at 0.191. 

### 2.2.

Calculate the weights of the mean-variance optimized portfolio, also called the tangency portfolio.

* Display a table indexed by each investment, with the optimal weights in one column and the Sharpe ratios in another column.

* Do the investments with the best Sharpe ratios tend to get the biggest weights?

#### Note:
To estimate the optimal weights, consider using the provided function below.

In [12]:
def optimized_weights(returns,dropna=True,scale_cov=1):
    if dropna:
        returns = returns.dropna()

    covmat_full = returns.cov()
    covmat_diag = np.diag(np.diag(covmat_full))
    covmat = scale_cov * covmat_full + (1-scale_cov) * covmat_diag

    weights = np.linalg.solve(covmat,returns.mean())
    weights = weights / weights.sum()

    if returns.mean() @ weights < 0:
        weights = -weights

    return pd.DataFrame(weights, index=returns.columns)

weights = optimized_weights(rets).rename(
    columns={0: "Optimal Weight"}
)

allocation = weights.join(
    summary[["Annualized Sharpe Ratio"]]
)

display(allocation.round(4))

print(f"Net weight: {weights['Optimal Weight'].sum():.2%}")
print(f"Gross exposure: {weights['Optimal Weight'].abs().sum():.2%}")


,Optimal Weight,Annualized Sharpe Ratio
AAPL,0.3532,0.8952
NVDA,0.8463,1.4202
MSFT,0.3013,0.9894
GOOGL,0.5513,0.9392
AMZN,0.1292,0.8581
META,0.1002,0.6491
TSLA,0.2005,0.7548
AVGO,0.4543,1.0297
BRK/B,1.6962,0.7164
LLY,0.9050,1.0107


Net weight: 100.00%
Gross exposure: 1007.53%


The investments with the highest standalone Sharpe ratios do not consistently receive the largest weights. NVDA has the highest Sharpe ratio and a large allication however BRKB receives the largest long allocation. 

### 2.3.

Report the following performance statistics of the portfolio achieved with the optimized weights calculated above.
* mean
* volatility
* Sharpe

(Annualize all three statistics.)

In [13]:
portfolio_returns = rets @ weights["Optimal Weight"]

portfolio_returns = portfolio_returns.to_frame(
    "Optimized Portfolio"
)

portfolio_summary = calc_return_metrics(
    portfolio_returns,
    as_df=True,
    adj=FREQ
)

display(
    portfolio_summary[[
        "Annualized Return",
        "Annualized Volatility",
        "Annualized Sharpe Ratio"
    ]].round(4)
)

,Annualized Return,Annualized Volatility,Annualized Sharpe Ratio
Optimized Portfolio,1.0201,0.5222,1.9536


### 2.4.

Try dropping the asset which had the biggest short position from the investment set. Re-run the optimization. What do you think of these new weights compared to the original optimized weights?

What is going on?

In [14]:
biggest_short = weights["Optimal Weight"].idxmin()
print(f"Dropped asset: {biggest_short}")

rets_reduced = rets.drop(columns=biggest_short)

new_weights = optimized_weights(rets_reduced).rename(
    columns={0: "New Weight"}
)

# The removed asset has zero weight in the new portfolio.
weight_comparison = weights.join(new_weights).fillna(0)

weight_comparison["Change"] = (
    weight_comparison["New Weight"]
    - weight_comparison["Optimal Weight"]
)

display(weight_comparison.round(4))

new_portfolio_returns = (
    rets_reduced @ new_weights["New Weight"]
).to_frame("Without " + biggest_short)

new_portfolio_summary = calc_return_metrics(
    new_portfolio_returns,
    as_df=True,
    adj=FREQ
)

performance_comparison = pd.concat([
    portfolio_summary,
    new_portfolio_summary
])

display(
    performance_comparison[[
        "Annualized Return",
        "Annualized Volatility",
        "Annualized Sharpe Ratio"
    ]].round(4)
)

print(
    f"Original gross exposure: "
    f"{weights['Optimal Weight'].abs().sum():.2%}"
)

print(
    f"New gross exposure: "
    f"{new_weights['New Weight'].abs().sum():.2%}"
)

Dropped asset: SPY


,Optimal Weight,New Weight,Change
AAPL,0.3532,0.0062,-0.3470
NVDA,0.8463,0.4049,-0.4414
MSFT,0.3013,-0.1131,-0.4145
GOOGL,0.5513,0.1765,-0.3748
AMZN,0.1292,-0.0344,-0.1636
META,0.1002,-0.0448,-0.1450
TSLA,0.2005,0.0235,-0.1770
AVGO,0.4543,0.1118,-0.3425
BRK/B,1.6962,0.0385,-1.6577
LLY,0.9050,0.4309,-0.4741


,Annualized Return,Annualized Volatility,Annualized Sharpe Ratio
Optimized Portfolio,1.0201,0.5222,1.9536
Without SPY,0.4539,0.2725,1.6657


Original gross exposure: 1007.53%
New gross exposure: 138.47%


Here SPY gave the optimizer a broad market hedge against its long positions. That allows it to construct a heavily leveraged portfolio. Removing it causes BRKB to fall while MSFT, META, and AMZN become short positions. Here the sharpe ratio falls as we add that constraint. 

****